# Tier 1A Track A — HATFormer Diagnosis-Driven Zero-Training Decoding (AraMS-28k)

Runs the complete zero-training decoding ablation on the fine-tuned HATFormer checkpoint (`hatformer_best_weights`) on **val** (`1,486` lines) and/or **test** (`6,744` lines):
1. `--blank-gate` (skip corrupt empty/margin crops)
2. `--width-cap` (aspect-scaled `max_new_tokens` fit strictly on `train`)
3. `--rep-penalty` sweep (`repetition_penalty` in `{1.0, 1.10, 1.15, 1.20}`, `length_penalty` in `{1.0, 1.1}`, NO global `no_repeat_ngram_size=3`)
4. `--loop-fallback` (surgical re-decode of flagged loop lines only with `rep_penalty=1.35`, `no_repeat_ngram_size=4`)

**Kaggle setup:** Accelerator = `GPU T4 x2`, Internet = `ON`. Optionally attach `hatformer_best_weights.zip` (or your previous Kaggle notebook output) and `AraMS-28k-HTR` as Kaggle Inputs; otherwise `Archatext/hatformer-arams28k` / Zenodo will be fetched automatically.

In [ ]:
# ============================ 1. SETTINGS ============================
import os, glob, shutil, zipfile, subprocess

SMOKE_TEST      = False   # True = quick --limit 32 sanity check; False = full val + test ablation
RUN_VAL_SWEEP   = True    # Recommended: validate best (rep_penalty, length_penalty) on val (1,486 lines) first
RUN_TEST_SWEEP  = True    # Run full ablation on test (6,744 lines)
REPO_URL        = "https://github.com/AMYasserF/arams28k-baseline.git"
REPO_BRANCH     = "main"
DATASET_ZIP_URL = "https://zenodo.org/records/21499649/files/AraMS-28k-HTR.zip?download=1"
CKPT_HF_REPO    = "Archatext/hatformer-arams28k"  # Fallback only if local/attached fine-tuned checkpoint not found
CKPT_OVERRIDE   = ""      # Optional explicit path to fine-tuned checkpoint directory (containing config.json + weights)
BATCH_SIZE      = 32

WORK = "/kaggle/working" if os.path.isdir("/kaggle") else os.getcwd()
TMP  = "/kaggle/temp" if os.path.isdir("/kaggle") else "/tmp/kaggle_tier1a"
os.makedirs(TMP, exist_ok=True)
REPO_DIR = f"{WORK}/arams28k-baseline"
OUT_ROOT = f"{WORK}/tier1a_hatformer_results"
os.makedirs(OUT_ROOT, exist_ok=True)

def sh(cmd, check=True, cwd=None):
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", cwd=cwd, text=True, bufsize=1,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"command failed (exit {rc}): {cmd}")
    return rc

def find_files(root, name, max_depth=7):
    hits = []
    if not os.path.isdir(root): return hits
    base = root.rstrip("/").count("/")
    for dp, dn, fn in os.walk(root):
        if dp.count("/") - base >= max_depth: dn[:] = []
        if name in fn: hits.append(os.path.join(dp, name))
    return sorted(hits)

In [ ]:
# ============================ 2. GPU & REPO CHECK ============================
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

if os.path.isdir(f"{REPO_DIR}/.git"):
    sh(f"git -C {REPO_DIR} fetch origin {REPO_BRANCH} && git -C {REPO_DIR} reset --hard origin/{REPO_BRANCH}")
else:
    sh(f"git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}")

sh("python -c 'import yaml, pandas, PIL, rapidfuzz' 2>/dev/null || pip install -q pyyaml pandas pillow rapidfuzz")
SCRIPT = f"{REPO_DIR}/scripts/models/hatformer/tier1a_decode.py"
assert os.path.isfile(SCRIPT), f"Missing {SCRIPT} — push latest scripts/models/hatformer/tier1a_decode.py first."

In [ ]:
# ============================ 3. LOCATE DATA & CHECKPOINT ============================
def locate_data():
    for root in ("/kaggle/input", f"{TMP}/data", f"{REPO_DIR}/data"):
        for m in find_files(root, "metadata.csv"):
            if os.path.isdir(os.path.join(os.path.dirname(m), "images")):
                return os.path.dirname(m)
    return None

DATA_DIR = locate_data()
if DATA_DIR is None:
    os.makedirs(f"{TMP}/data", exist_ok=True)
    sh(f"wget -nv -c -O {TMP}/AraMS-28k-HTR.zip '{DATASET_ZIP_URL}'")
    sh(f"unzip -q -o {TMP}/AraMS-28k-HTR.zip -d {TMP}/data && rm -f {TMP}/AraMS-28k-HTR.zip")
    DATA_DIR = locate_data()
assert DATA_DIR, "Could not locate AraMS-28k-HTR metadata.csv + images/"
print("DATA_DIR =", DATA_DIR)

def locate_checkpoint():
    if CKPT_OVERRIDE and os.path.isdir(CKPT_OVERRIDE):
        return CKPT_OVERRIDE
    # Check for unzipped best weights or hatformer_best_weights.zip in /kaggle/input or WORK
    for root in ("/kaggle/input", WORK):
        for z in find_files(root, "hatformer_best_weights.zip"):
            ext_dir = f"{TMP}/hatformer_best_weights"
            if not os.path.isfile(f"{ext_dir}/config.json"):
                os.makedirs(ext_dir, exist_ok=True)
                with zipfile.ZipFile(z, "r") as zf:
                    zf.extractall(ext_dir)
            return ext_dir
        for cfgp in find_files(root, "config.json"):
            d = os.path.dirname(cfgp)
            fs = os.listdir(d)
            if any(f.endswith((".safetensors", ".bin")) for f in fs) and (
                "best" in d.lower() or "hatformer" in d.lower()
            ):
                return d
    return None

CKPT_DIR = locate_checkpoint()
if CKPT_DIR is None:
    from huggingface_hub import snapshot_download
    CKPT_DIR = snapshot_download(CKPT_HF_REPO, local_dir=f"{TMP}/hf_hatformer")
    print("Downloaded HF checkpoint fallback:", CKPT_DIR)
print("CKPT_DIR =", CKPT_DIR, "| files =", sorted(os.listdir(CKPT_DIR)))

TOK_PATH = f"{CKPT_DIR}/tokenizer.json"
if not os.path.isfile(TOK_PATH):
    cands = [c for c in find_files("/kaggle/input", "tokenizer.json") if os.path.getsize(c) > 100_000]
    if cands:
        TOK_PATH = cands[0]
    else:
        from huggingface_hub import hf_hub_download
        TOK_PATH = hf_hub_download(CKPT_HF_REPO, "tokenizer.json", local_dir=f"{TMP}/tok_hf")
print("TOK_PATH =", TOK_PATH)

In [ ]:
# ============================ 4. SELF-TEST & CALIBRATION VERIFICATION ============================
sh(f"python -u {SCRIPT} --self-test --data-dir '{DATA_DIR}'")

In [ ]:
# ============================ 5. VAL & TEST TIER 1A ABLATION ============================
limit_flag = "--limit 32" if SMOKE_TEST else ""
common = f"--checkpoint '{CKPT_DIR}' --tokenizer '{TOK_PATH}' --data-dir '{DATA_DIR}' --batch-size {BATCH_SIZE} {limit_flag}"

if RUN_VAL_SWEEP:
    sh(f"python -u {SCRIPT} {common} --split val --run-ablation --output-dir '{OUT_ROOT}/val_ablation'")

if RUN_TEST_SWEEP:
    sh(f"python -u {SCRIPT} {common} --split test --run-ablation --output-dir '{OUT_ROOT}/test_ablation'")

In [ ]:
# ============================ 6. PACKAGE RESULTS FOR DOWNLOAD ============================
zip_path = shutil.make_archive(f"{WORK}/tier1a_hatformer_results", "zip", root_dir=OUT_ROOT)
print(f"Saved archive: {zip_path} ({os.path.getsize(zip_path) / 1e6:.2f} MB)")
for md in sorted(glob.glob(f"{OUT_ROOT}/**/ablation_summary.md", recursive=True)):
    print("\n--- " + md + " ---")
    with open(md, encoding="utf-8") as f:
        print(f.read())